In [129]:
# importing relevant librarise for the project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly as pl
import seaborn as sns
import re

In [130]:
df = pd.read_csv("Data\\cleaned_booking.csv", index_col=0)
df = df.reset_index(drop=True)
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,resort hotel,0,342,2015,july,27,1,0,0,2,...,no deposit,NaN,NaN,0,transient,0.0,0,0,checkout,712015
1,resort hotel,0,737,2015,july,27,1,0,0,2,...,no deposit,NaN,NaN,0,transient,0.0,0,0,checkout,712015
2,resort hotel,0,7,2015,july,27,1,0,1,1,...,no deposit,NaN,NaN,0,transient,75.0,0,0,checkout,722015
3,resort hotel,0,13,2015,july,27,1,0,1,1,...,no deposit,304.0,NaN,0,transient,75.0,0,0,checkout,722015
4,resort hotel,0,14,2015,july,27,1,0,2,2,...,no deposit,240.0,NaN,0,transient,98.0,0,1,checkout,732015


In [131]:
columns_data = df.isna().sum().sort_values(ascending=False)
empty_column = {col: val for col, val in columns_data.items() if val > 0}
empty_column

{'company': 112593, 'agent': 16340, 'country': 488, 'children': 4}

now handling missing values in the data. different techniques for handling missning numerical values:

1. imputation methods
   mean/meadian/mode imputation: df["colname"].fillna(df["colname"].mean/median/mode(), inplace=True)
   forward fill: df["colname"].fillna(methode='ffill', inplace=True)
   backward fill: df["colname"].fillna(methode='bfill', inplace=True)
2. interpolation: Interpolation predicts the values which are present within the known data range.

Method | Use Case | Pros | Cons

1. Drop Rows (df.dropna()) | Small % of rows missing randomly | Simple, clean | Data loss, may remove valuable info
2. Drop Columns (df.drop(columns=...)) | If entire column is mostly missing | Reduces dimensionality | Can lose features that may have been useful
3. Fill with a Constant (fillna(0), 'unknown') | Categorical/text data or meaningful defaults | Fast, clear behavior | Can introduce bias, flatten distribution
4. Mean/Median/Mode Imputation | Numerical features (mean), Categorical (mode) | Preserves dataset size | Distorts variance, weakens relationships
5. Forward/Backward Fill (ffill, bfill) | Time-series or ordered data | Keeps continuity | Can propagate incorrect values
6. Interpolation | Time series or numerical sequences | Smart estimate based on trends | Not suitable for all data types
7. Model-based Imputation | Use ML models to predict missing values | More accurate | Complex, needs tuning
8. KNN Imputation | Small to medium datasets with similar rows | Can capture non-linear relations | Slow, sensitive to outliers
9. Indicator for Missingness | Add binary flag column is_missing | Preserves info about missingness itself | Adds complexity
10.   Leave As Is (np.nan) | If your algorithm handles missing (e.g., XGBoost, CatBoost) | Minimal preprocessing | Not suitable for most traditional ML models


In [132]:
agent_list = df['agent'].unique().tolist()


In [ ]:
def handle_empty_data(df, empty_column):

    # if more than 50% data is missing remove the column entirely:

    threshold = len(df) * 0.4
    df = df.dropna(thresh=threshold, axis=1)
    
    # replacing numerical missing values with -1 indication data not available.
    for col, val in empty_column.items():
        if col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].fillna(-1)


    df = df.dropna()
    return df

In [134]:
handle_empty_data(df, empty_column)

C:\Users\Kartik\AppData\Local\Temp\ipykernel_13292\1174183580.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna(-1)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,resort hotel,0,342,2015,july,27,1,0,0,2,...,3,no deposit,-1.0,0,transient,0.00,0,0,checkout,712015
1,resort hotel,0,737,2015,july,27,1,0,0,2,...,4,no deposit,-1.0,0,transient,0.00,0,0,checkout,712015
2,resort hotel,0,7,2015,july,27,1,0,1,1,...,0,no deposit,-1.0,0,transient,75.00,0,0,checkout,722015
3,resort hotel,0,13,2015,july,27,1,0,1,1,...,0,no deposit,304.0,0,transient,75.00,0,0,checkout,722015
4,resort hotel,0,14,2015,july,27,1,0,2,2,...,0,no deposit,240.0,0,transient,98.00,0,1,checkout,732015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,city hotel,0,23,2017,august,35,30,2,5,2,...,0,no deposit,394.0,0,transient,96.14,0,0,checkout,962017
119386,city hotel,0,102,2017,august,35,31,2,5,3,...,0,no deposit,9.0,0,transient,225.43,0,2,checkout,972017
119387,city hotel,0,34,2017,august,35,31,2,5,2,...,0,no deposit,9.0,0,transient,157.71,0,4,checkout,972017
119388,city hotel,0,109,2017,august,35,31,2,5,2,...,0,no deposit,89.0,0,transient,104.40,0,0,checkout,972017


In [135]:
df.isna().sum().sort_values(ascending=False)

company                           112593
agent                              16340
country                              488
children                               4
arrival_date_month                     0
arrival_date_week_number               0
hotel                                  0
is_canceled                            0
stays_in_weekend_nights                0
arrival_date_day_of_month              0
adults                                 0
stays_in_week_nights                   0
babies                                 0
meal                                   0
lead_time                              0
arrival_date_year                      0
distribution_channel                   0
market_segment                         0
previous_bookings_not_canceled         0
is_repeated_guest                      0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
previous_cancellations                 0
deposit_type    

In [136]:
obj_data = df.select_dtypes(include="object").columns.tolist()
obj_data

['hotel',
 'arrival_date_month',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'reserved_room_type',
 'assigned_room_type',
 'deposit_type',
 'customer_type',
 'reservation_status']

In [137]:
distinct_values = {obj: df[obj].nunique() for obj in obj_data}
distinct_values

{'hotel': 2,
 'arrival_date_month': 12,
 'meal': 5,
 'country': 177,
 'market_segment': 8,
 'distribution_channel': 5,
 'reserved_room_type': 10,
 'assigned_room_type': 12,
 'deposit_type': 3,
 'customer_type': 4,
 'reservation_status': 3}

since ML models doesnt work on categorical data so handling cateforical data based on nomial and ordinal data. also column countries will remains as is as there are multiple entries.

we have single ordinal data where order matter which is month column


In [138]:
distinct_names= {col: df[col].unique() for col in obj_data}
distinct_names

{'hotel': array(['resort hotel', 'city hotel'], dtype=object),
 'arrival_date_month': array(['july', 'august', 'september', 'october', 'november', 'december',
        'january', 'february', 'march', 'april', 'may', 'june'],
       dtype=object),
 'meal': array(['bb', 'fb', 'hb', 'sc', 'undefined'], dtype=object),
 'country': array(['prt', 'gbr', 'usa', 'esp', 'irl', 'fra', nan, 'rou', 'nor', 'omn',
        'arg', 'pol', 'deu', 'bel', 'che', 'cn', 'grc', 'ita', 'nld',
        'dnk', 'rus', 'swe', 'aus', 'est', 'cze', 'bra', 'fin', 'moz',
        'bwa', 'lux', 'svn', 'alb', 'ind', 'chn', 'mex', 'mar', 'ukr',
        'smr', 'lva', 'pri', 'srb', 'chl', 'aut', 'blr', 'ltu', 'tur',
        'zaf', 'ago', 'isr', 'cym', 'zmb', 'cpv', 'zwe', 'dza', 'kor',
        'cri', 'hun', 'are', 'tun', 'jam', 'hrv', 'hkg', 'irn', 'geo',
        'and', 'gib', 'ury', 'jey', 'caf', 'cyp', 'col', 'ggy', 'kwt',
        'nga', 'mdv', 'ven', 'svk', 'fji', 'kaz', 'pak', 'idn', 'lbn',
        'phl', 'sen', 'syc', 'a

In [139]:
months_ordinal = {"january": 1, "feburary"}

SyntaxError: ':' expected after dictionary key (2649511200.py, line 1)